# Importando bibliotecas

In [6]:
import pandas as pd
from sklearn.linear_model import LinearRegression 
import numpy as np


# Abrindo os dados

In [8]:
# Caminho para o arquivo Parquet
caminho_parquet = r"C:\Users\Pichau\INMET\TODAS_ESTACOES\TODAS_ESTACOES_CONCATENADO.parquet"

# Ler o arquivo Parquet
df = pd.read_parquet(caminho_parquet)

# Verificando os dados

In [10]:
df.columns

Index(['Dt_Hr', 'timestamp', 'Lat', 'Long', 'Alt', 'Precip', 'Pres_Atm',
       'Pres_Atm_max', 'Pres_Atm_min', 'Rad', 'Temp_Amb', 'Pto_Orv',
       'Temp_Amb_max', 'Temp_Amb_min', 'Pto_Orv_max', 'Pto_Orv_min',
       'Umidade_max', 'Umidade_min', 'Umidade', 'Vento_dir', 'Vento_raj',
       'Vento_vel', 'ESTACAO', 'estacao_do_ano', 'hora', 'turno',
       'PRECIPITAÇÃO TOTAL primeira hora anterior',
       'PRECIPITAÇÃO TOTAL segunda hora anterior',
       'PRECIPITAÇÃO TOTAL terceira hora anterior',
       'VARIACAO PRIMEIRA HORA ANTERIOR PRESSAO',
       'VARIACAO SEGUNDA HORA ANTERIOR PRESSAO',
       'VARIACAO TERCEIRA HORA ANTERIOR PRESSAO',
       'VARIACAO QUARTA HORA ANTERIOR PRESSAO',
       'VARIACAO QUINTA HORA ANTERIOR PRESSAO',
       'VARIACAO SEXTA HORA ANTERIOR PRESSAO',
       'VARIACAO PRIMEIRA HORA ANTERIOR TEMPERATURA AMBIENTE',
       'VARIACAO SEGUNDA HORA ANTERIOR TEMPERATURA AMBIENTE',
       'VARIACAO TERCEIRA HORA ANTERIOR TEMPERATURA AMBIENTE',
       'VARIAC

In [11]:
df.head(2)

,Dt_Hr,timestamp,Lat,Long,Alt,Precip,Pres_Atm,Pres_Atm_max,Pres_Atm_min,Rad,...,VARIACAO SEXTA HORA ANTERIOR PRESSAO,VARIACAO PRIMEIRA HORA ANTERIOR TEMPERATURA AMBIENTE,VARIACAO SEGUNDA HORA ANTERIOR TEMPERATURA AMBIENTE,VARIACAO TERCEIRA HORA ANTERIOR TEMPERATURA AMBIENTE,VARIACAO PRIMEIRA HORA ANTERIOR TEMPERATURA ORVALHO,VARIACAO SEGUNDA HORA ANTERIOR TEMPERATURA ORVALHO,VARIACAO TERCEIRA HORA ANTERIOR TEMPERATURA ORVALHO,VARIACAO PRIMEIRA HORA ANTERIOR UMIDADE,VARIACAO SEGUNDA HORA ANTERIOR UMIDADE,VARIACAO TERCEIRA HORA ANTERIOR UMIDAD
0,2007-05-18 00:00:00,1.179457e+09,-22.988333,-43.190278,42.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,...,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN
1,2007-05-18 01:00:00,1.179461e+09,-22.988333,-43.190278,42.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,...,NaN,0.0,0.0,NaN,0.0,0.0,NaN,0.0,0.0,NaN


In [12]:
df['Dt_Hr'].min()


Timestamp('2002-11-08 00:00:00')

In [13]:
df['Dt_Hr'].max()

Timestamp('2024-11-30 23:00:00')

# Estação do ano

In [15]:
df['Dt_Hr'] = pd.to_datetime(df['Dt_Hr'])

def pega_estacao(Data):
    dia = Data.day
    mes = Data.month
    
    if (mes == 12 and dia >= 21) or (mes in [1, 2]) or (mes == 3 and dia <= 20):
        return 'Verão'
    elif (mes == 3 and dia >= 21) or (mes in [4, 5]) or (mes == 6 and dia <= 20):
        return 'Outono'
    elif (mes == 6 and dia >= 21) or (mes in [7, 8]) or (mes == 9 and dia <= 20):
        return 'Inverno'
    elif (mes == 9 and dia >= 21) or (mes in [10, 11]) or (mes == 12 and dia <= 20):
        return 'Primavera'

# Data das estações: https://www.todamateria.com.br/as-estacoes-do-ano/

# Criando a nova coluna com as estações do ano
df['estacao_do_ano'] = df['Dt_Hr'].apply(pega_estacao)

caminho_parquet = r"C:\Users\Pichau\INMET\TODAS_ESTACOES\TODAS_ESTACOES_CONCATENADO.parquet"

df.to_parquet(caminho_parquet, index=False)

In [16]:
df.head(1)

,Dt_Hr,timestamp,Lat,Long,Alt,Precip,Pres_Atm,Pres_Atm_max,Pres_Atm_min,Rad,...,VARIACAO SEXTA HORA ANTERIOR PRESSAO,VARIACAO PRIMEIRA HORA ANTERIOR TEMPERATURA AMBIENTE,VARIACAO SEGUNDA HORA ANTERIOR TEMPERATURA AMBIENTE,VARIACAO TERCEIRA HORA ANTERIOR TEMPERATURA AMBIENTE,VARIACAO PRIMEIRA HORA ANTERIOR TEMPERATURA ORVALHO,VARIACAO SEGUNDA HORA ANTERIOR TEMPERATURA ORVALHO,VARIACAO TERCEIRA HORA ANTERIOR TEMPERATURA ORVALHO,VARIACAO PRIMEIRA HORA ANTERIOR UMIDADE,VARIACAO SEGUNDA HORA ANTERIOR UMIDADE,VARIACAO TERCEIRA HORA ANTERIOR UMIDAD
0,2007-05-18,1.179457e+09,-22.988333,-43.190278,42.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,...,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN


# TURNO




In [18]:
def pega_turno(hora):
    if 5 <= hora < 12:
        return 'Manhã'
    elif 12 <= hora < 18:
        return 'Tarde'
    else:
        return 'Noite'
        
df['hora'] = df['Dt_Hr'].dt.hour

df['turno'] = df['hora'].apply(pega_turno)

caminho_parquet = r"C:\Users\Pichau\INMET\TODAS_ESTACOES\TODAS_ESTACOES_CONCATENADO.parquet"

df.to_parquet(caminho_parquet, index=False)


# PRECIPITAÇÃO HORA ANTERIOR 

In [20]:
df = df.sort_values(by=['ESTACAO', 'Dt_Hr'])

df['PRECIPITAÇÃO TOTAL primeira hora anterior'] = df.groupby('ESTACAO')['Precip'].shift(1)

df['PRECIPITAÇÃO TOTAL segunda hora anterior'] = df.groupby('ESTACAO')['Precip'].shift(2)

df['PRECIPITAÇÃO TOTAL terceira hora anterior'] = df.groupby('ESTACAO')['Precip'].shift(3)


caminho_parquet = r"C:\Users\Pichau\INMET\TODAS_ESTACOES\TODAS_ESTACOES_CONCATENADO.parquet"

df.to_parquet(caminho_parquet, index=False)


# Radiação Solar

In [18]:
df = df.sort_values(by=['ESTACAO', 'Dt_Hr'])

df['RADIAÇÃO PRIMEIRA HORA ANTERIOR'] = df.groupby('ESTACAO')['Rad'].shift(1)

df['RADIAÇÃO SEGUNDA HORA ANTERIOR'] = df.groupby('ESTACAO')['Rad'].shift(2)

df['RADIAÇÃO TERCEIRA HORA ANTERIOR'] = df.groupby('ESTACAO')['Rad'].shift(3)

# DIREÇÃO DO VENTO HORA ANTERIOR 

In [22]:


df = df.sort_values(by=['ESTACAO', 'Dt_Hr'])

df['DIRECAO DO VENTO PRIMEIRA HORA ANTERIOR'] = df.groupby('ESTACAO')['Vento_dir'].shift(1)


df['DIRECAO DO VENTO SEGUNDA HORA ANTERIOR'] = df.groupby('ESTACAO')['Vento_dir'].shift(2)


df['DIRECAO DO VENTO TERCEIRA HORA ANTERIOR'] = df.groupby('ESTACAO')['Vento_dir'].shift(3)


df['DIRECAO DO VENTO QUARTA HORA ANTERIOR'] = df.groupby('ESTACAO')['Vento_dir'].shift(4)

df['DIRECAO DO VENTO QUINTA HORA ANTERIOR'] = df.groupby('ESTACAO')['Vento_dir'].shift(5)

df['DIRECAO DO VENTO SEXTA HORA ANTERIOR'] = df.groupby('ESTACAO')['Vento_dir'].shift(6)


caminho_parquet = r"C:\Users\Pichau\INMET\TODAS_ESTACOES\TODAS_ESTACOES_CONCATENADO.parquet"

df.to_parquet(caminho_parquet, index=False)

# Velocidade do vento hora anteior

In [ ]:
df = df.sort_values(by=['ESTACAO', 'Dt_Hr'])

df['VELOCIDADE DO VENTO PRIMEIRA HORA ANTERIOR'] = df.groupby('ESTACAO')['Vento_vel'].shift(1)


df['VELOCIDADE DO VENTO SEGUNDA HORA ANTERIOR'] = df.groupby('ESTACAO')['Vento_vel'].shift(2)


df['VELOCIDADE DO VENTO TERCEIRA HORA ANTERIOR'] = df.groupby('ESTACAO')['Vento_vel'].shift(3)



caminho_parquet = r"C:\Users\Pichau\INMET\TODAS_ESTACOES\TODAS_ESTACOES_CONCATENADO.parquet"

df.to_parquet(caminho_parquet, index=False)

# PRESSÃO ATMOSFÉRICA HORA ANTERIOR

In [24]:
df.head(1)

,Dt_Hr,timestamp,Lat,Long,Alt,Precip,Pres_Atm,Pres_Atm_max,Pres_Atm_min,Rad,...,VARIACAO TERCEIRA HORA ANTERIOR TEMPERATURA ORVALHO,VARIACAO PRIMEIRA HORA ANTERIOR UMIDADE,VARIACAO SEGUNDA HORA ANTERIOR UMIDADE,VARIACAO TERCEIRA HORA ANTERIOR UMIDAD,DIRECAO DO VENTO PRIMEIRA HORA ANTERIOR,DIRECAO DO VENTO SEGUNDA HORA ANTERIOR,DIRECAO DO VENTO TERCEIRA HORA ANTERIOR,DIRECAO DO VENTO QUARTA HORA ANTERIOR,DIRECAO DO VENTO QUINTA HORA ANTERIOR,DIRECAO DO VENTO SEXTA HORA ANTERIOR
0,2007-05-18,1.179457e+09,-22.988333,-43.190278,42.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:

df['VARIACAO PRIMEIRA HORA ANTERIOR PRESSAO'] = (
    df['Pres_Atm_max'] -
    df['Pres_Atm_min'] 
)



In [26]:
df = df.sort_values(by=['ESTACAO', 'Dt_Hr'])

df['VARIACAO SEGUNDA HORA ANTERIOR PRESSAO'] = df.groupby('ESTACAO')['VARIACAO PRIMEIRA HORA ANTERIOR PRESSAO'].shift(1)

df['VARIACAO TERCEIRA HORA ANTERIOR PRESSAO'] = df.groupby('ESTACAO')['VARIACAO PRIMEIRA HORA ANTERIOR PRESSAO'].shift(2)

df['VARIACAO QUARTA HORA ANTERIOR PRESSAO'] = df.groupby('ESTACAO')['VARIACAO PRIMEIRA HORA ANTERIOR PRESSAO'].shift(3)

df['VARIACAO QUINTA HORA ANTERIOR PRESSAO'] = df.groupby('ESTACAO')['VARIACAO PRIMEIRA HORA ANTERIOR PRESSAO'].shift(4)

df['VARIACAO SEXTA HORA ANTERIOR PRESSAO'] = df.groupby('ESTACAO')['VARIACAO PRIMEIRA HORA ANTERIOR PRESSAO'].shift(5)


caminho_parquet = r"C:\Users\Pichau\INMET\TODAS_ESTACOES\TODAS_ESTACOES_CONCATENADO.parquet"

df.to_parquet(caminho_parquet, index=False)

# Verificação da variavel radição total. 

In [28]:


# Supondo que o seu DataFrame se chama 'df'
count_nan = df['Rad'].isna().sum()
count_non_nan = df['Rad'].notna().sum()

# Calculando os valores relativos
total_values = len(df['Rad'])
relative_nan = count_nan / total_values * 100
relative_non_nan = count_non_nan / total_values * 100

print(f"Número de valores NaN: {count_nan}")
print(f"Número de valores não nulos: {count_non_nan}")
print(f"Proporção de valores NaN: {relative_nan:.2f}%")
print(f"Proporção de valores não nulos: {relative_non_nan:.2f}%")


Número de valores NaN: 78927
Número de valores não nulos: 486945
Proporção de valores NaN: 13.95%
Proporção de valores não nulos: 86.05%


# VARIAÇÃO DE TEMPERATURA AMBIENTE

In [30]:
# TEMPERATURA DO AR - BULBO SECO, HORARIA (°C) 
# TEMPERATURA DO PONTO DE ORVALHO (°C) 
# TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C) 
# TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C) 
# TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C) 
# TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C) 


In [31]:
df.columns

Index(['Dt_Hr', 'timestamp', 'Lat', 'Long', 'Alt', 'Precip', 'Pres_Atm',
       'Pres_Atm_max', 'Pres_Atm_min', 'Rad', 'Temp_Amb', 'Pto_Orv',
       'Temp_Amb_max', 'Temp_Amb_min', 'Pto_Orv_max', 'Pto_Orv_min',
       'Umidade_max', 'Umidade_min', 'Umidade', 'Vento_dir', 'Vento_raj',
       'Vento_vel', 'ESTACAO', 'estacao_do_ano', 'hora', 'turno',
       'PRECIPITAÇÃO TOTAL primeira hora anterior',
       'PRECIPITAÇÃO TOTAL segunda hora anterior',
       'PRECIPITAÇÃO TOTAL terceira hora anterior',
       'VARIACAO PRIMEIRA HORA ANTERIOR PRESSAO',
       'VARIACAO SEGUNDA HORA ANTERIOR PRESSAO',
       'VARIACAO TERCEIRA HORA ANTERIOR PRESSAO',
       'VARIACAO QUARTA HORA ANTERIOR PRESSAO',
       'VARIACAO QUINTA HORA ANTERIOR PRESSAO',
       'VARIACAO SEXTA HORA ANTERIOR PRESSAO',
       'VARIACAO PRIMEIRA HORA ANTERIOR TEMPERATURA AMBIENTE',
       'VARIACAO SEGUNDA HORA ANTERIOR TEMPERATURA AMBIENTE',
       'VARIACAO TERCEIRA HORA ANTERIOR TEMPERATURA AMBIENTE',
       'VARIAC

In [32]:
df['VARIACAO PRIMEIRA HORA ANTERIOR TEMPERATURA AMBIENTE'] = (
    df['Temp_Amb_max'] -
    df['Temp_Amb_min'] 
)

df['VARIACAO SEGUNDA HORA ANTERIOR TEMPERATURA AMBIENTE'] = df.groupby('ESTACAO')['VARIACAO PRIMEIRA HORA ANTERIOR TEMPERATURA AMBIENTE'].shift(1)

df['VARIACAO TERCEIRA HORA ANTERIOR TEMPERATURA AMBIENTE'] = df.groupby('ESTACAO')['VARIACAO PRIMEIRA HORA ANTERIOR TEMPERATURA AMBIENTE'].shift(2)

caminho_parquet = r"C:\Users\Pichau\INMET\TODAS_ESTACOES\TODAS_ESTACOES_CONCATENADO.parquet"

df.to_parquet(caminho_parquet, index=False)


# VARIAÇÃO DE TEMPERATURA ORVALHO

In [34]:
df['VARIACAO PRIMEIRA HORA ANTERIOR TEMPERATURA ORVALHO'] = (
    df['Pto_Orv_max'] -
    df['Pto_Orv_min'] 
) 

df['VARIACAO SEGUNDA HORA ANTERIOR TEMPERATURA ORVALHO'] = df.groupby('ESTACAO')['VARIACAO PRIMEIRA HORA ANTERIOR TEMPERATURA ORVALHO'].shift(1)

df['VARIACAO TERCEIRA HORA ANTERIOR TEMPERATURA ORVALHO'] = df.groupby('ESTACAO')['VARIACAO PRIMEIRA HORA ANTERIOR TEMPERATURA ORVALHO'].shift(2)

caminho_parquet = r"C:\Users\Pichau\INMET\TODAS_ESTACOES\TODAS_ESTACOES_CONCATENADO.parquet"

df.to_parquet(caminho_parquet, index=False)



In [35]:
df.head(1)

,Dt_Hr,timestamp,Lat,Long,Alt,Precip,Pres_Atm,Pres_Atm_max,Pres_Atm_min,Rad,...,VARIACAO TERCEIRA HORA ANTERIOR TEMPERATURA ORVALHO,VARIACAO PRIMEIRA HORA ANTERIOR UMIDADE,VARIACAO SEGUNDA HORA ANTERIOR UMIDADE,VARIACAO TERCEIRA HORA ANTERIOR UMIDAD,DIRECAO DO VENTO PRIMEIRA HORA ANTERIOR,DIRECAO DO VENTO SEGUNDA HORA ANTERIOR,DIRECAO DO VENTO TERCEIRA HORA ANTERIOR,DIRECAO DO VENTO QUARTA HORA ANTERIOR,DIRECAO DO VENTO QUINTA HORA ANTERIOR,DIRECAO DO VENTO SEXTA HORA ANTERIOR
0,2007-05-18,1.179457e+09,-22.988333,-43.190278,42.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,...,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Variação de umidade. 

In [37]:
df['VARIACAO PRIMEIRA HORA ANTERIOR UMIDADE'] = (
    df['Umidade_max'] -
    df['Umidade_min'] 
) 

df['VARIACAO SEGUNDA HORA ANTERIOR UMIDADE' ] = df.groupby('ESTACAO')['VARIACAO PRIMEIRA HORA ANTERIOR UMIDADE'].shift(1)

df['VARIACAO TERCEIRA HORA ANTERIOR UMIDAD'] = df.groupby('ESTACAO')['VARIACAO PRIMEIRA HORA ANTERIOR UMIDADE'].shift(2)

caminho_parquet = r"C:\Users\Pichau\INMET\TODAS_ESTACOES\TODAS_ESTACOES_CONCATENADO.parquet"

df.to_parquet(caminho_parquet, index=False)


# VENTO 